In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import yfinance as yf

In [2]:
ticker = '^NSEI'
start_date = '2014-01-01'
end_date = '2025-01-01'

df = yf.download(ticker,start_date,end_date)
df

/tmp/ipykernel_2253/1502302891.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker,start_date,end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2014-01-02,6221.149902,6358.299805,6211.299805,6301.250000,158100
2014-01-03,6211.149902,6221.700195,6171.250000,6194.549805,139000
2014-01-06,6191.450195,6224.700195,6170.250000,6220.850098,118300
2014-01-07,6162.250000,6221.500000,6144.750000,6203.899902,138600
2014-01-08,6174.600098,6192.100098,6160.350098,6178.049805,146900
...,...,...,...,...,...
2024-12-24,23727.650391,23867.650391,23685.150391,23769.099609,177700
2024-12-26,23750.199219,23854.500000,23653.599609,23775.800781,177700


**A1 - STATIONARITY**

In [3]:
from statsmodels.tsa.stattools import adfuller

In [5]:
result = adfuller(df['Close']['^NSEI'])
print(result[1])

0.97864667584301


In [54]:
daily_returns_matrix = df['Close']['^NSEI'].pct_change().dropna()
daily_returns_matrix.dropna(inplace=True)
result = adfuller(daily_returns_matrix)
print(result[1])

7.288607985649486e-27


**A2 - LOOK AHEAD BIAS**

In [15]:
from sklearn.model_selection import train_test_split

In [38]:
returns = df['Close']['^NSEI'].pct_change().dropna()
returns = returns.reset_index(drop=True)


df_returns = pd.DataFrame()
df_returns['returns'] = returns.values
df_returns['lag1'] = df_returns['returns'].shift(1)
df_returns['target'] = (df_returns['returns'] > 0).astype(int)
df_returns.dropna(inplace=True)

df_returns

,returns,lag1,target
1,-0.004716,-0.003172,0
2,0.002004,-0.004716,1
3,-0.001012,0.002004,0
4,0.000503,-0.001012,1
5,0.016414,0.000503,1
...,...,...,...
2692,-0.001086,0.007035,0
2693,0.000950,-0.001086,1
2694,0.002661,0.000950,1
2695,-0.007076,0.002661,0


**A3 - WALK-FORWARD VALIDATION**

In [56]:
from sklearn.linear_model import LogisticRegression

In [57]:
X = df_returns[['lag1']].values
y = df_returns['target'].values

# Split 1: shuffled (wrong way)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
model = LogisticRegression()
model.fit(X_train, y_train)
print("Shuffled accuracy:", model.score(X_test, y_test))

# Split 2: time-ordered (correct way)
split = int(len(X) * 0.8)
X_train2, X_test2 = X[:split], X[split:]
y_train2, y_test2 = y[:split], y[split:]
model2 = LogisticRegression()
model2.fit(X_train2, y_train2)
print("Time-ordered accuracy:", model2.score(X_test2, y_test2))


Shuffled accuracy: 0.5425925925925926
Time-ordered accuracy: 0.562962962962963


In [59]:
accuracies = []
for i in range(1500, len(X), 50):
  X_train = X[:i]
  y_train = y[:i]
  X_test = X[i:i+50]
  y_test = y[i:i+50]
  model = LogisticRegression()
  model.fit(X_train, y_train)
  model.predict(X_test)
  accuracies.append(model.score(X_test, y_test))

print(np.mean(accuracies))

0.5580434782608696


**A4 - OVERFITTING IN FINANCE**

In [66]:
from sklearn.tree import DecisionTreeClassifier

X = df_returns[['lag1']]
y = df_returns['target']
X_train = X.iloc[0:1501]
y_train = y.iloc[0:1501]

model = DecisionTreeClassifier()
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 1.0
Test accuracy: 0.5372384937238494


In [67]:
model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)
print("Training accuracy:",model.score(X_train, y_train))
print("Test accuracy:",model.score(X.iloc[1501:], y.iloc[1501:]))

Training accuracy: 0.5489673550966022
Test accuracy: 0.5447698744769874
